# **1. Obtain from each model:** Results, execution time, parameters and CO2.

> **Note (SUSCOM version).** The data of this notebook now come from the checkpoints behind the paper tables
> (`USE_PAPER_CHECKPOINTS = True` in the first code cell): ETeR-X and the Amazon USEM/BERT models of
> `results/user_agg_summary_known.csv`, and the TripAdvisor USEM/BERT models retrained on the ETeR-X split
> (`results/tripadvisor_retrain_md5.csv`), plus the zero-shot baselines EmbeddingGemma/Qwen3-Embedding
> (`results/embedding_results*.csv`) and kNN-BoW. Emissions are in **grams** of CO2. Section 1 (users) is only valid for the
> old `best_models.csv` checkpoints; the efficiency figure is `output/bar_plot_co2_params.pdf`.


In [ ]:
# Import pandas
from src.experiments.Common import load_best_model
from matplotlib.ticker import FixedLocator
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os

baselines_path = "models/Baselines"

datasets = {"restaurants":["gijon", "barcelona", "madrid", "paris", "newyorkcity"],
            "pois":["barcelona", "madrid", "paris", "newyorkcity", "london"],
            "amazon":["fashion", "digital_music"]}

our_model="ETeR-X"

set_names = {"Amazon":"AM", "Pois":"POI", "Restaurants":"RST"}
subset_names = {"Digital_Music":"MSC", "Music":"MSC", "Fashion":"FSH", "Gijon":"GJN", "Gijón":"GJN", "Barcelona":"BCN", "Madrid":"MDR", "Newyorkcity":"NYC", "New York":"NYC", "Paris":"PRS", "London":"LND"}

models_cold = {"MOSTPOP2ITM":"MostPop", "BOW2ITM":our_model, "USEM2ITM":"USEM", "BERT2ITM":"BERT"}
models_baseline = {"MOSTPOP2ITM":"MostPop", 'EASEᴿ':"EASEᴿ", 'BiVAECF':"BiVAE", 'GridSearch_BPR':"BPR", "BOW2ITM":our_model, "USEM2ITM":"USEM", "BERT2ITM":"BERT"}  # BPR was computed all along (Baselines.py), but silently dropped from every table/figure until now

model_colors = {'BiVAE':"#FF708F", 'EASEᴿ':"#95ABB2", 'MostPop':'#AD858B', 'BPR':'#8D6E63', our_model:'#4CAF50', 'USEM':'#03A9F4', 'BERT':'#ffa726'}

# ---------------------------------------------------------------------------------------------
# Elements added for the paper (SUSCOM version):
#  * the zero-shot embedding baselines (EmbeddingGemma, Qwen3-Embedding) and kNN-BoW,
#  * the checkpoints behind the paper tables: ETeR-X and the Amazon USEM/BERT models listed in
#    results/user_agg_summary_known.csv, and the TripAdvisor USEM/BERT models retrained on the ETeR-X
#    split (results/tripadvisor_retrain_md5.csv) instead of the "best_models.csv" ones,
#  * emissions in grams of CO2.
# ---------------------------------------------------------------------------------------------
import csv

USE_PAPER_CHECKPOINTS = True   # False -> original behaviour (load_best_model on models/best_models.csv, per-model evaluation)
results_path = "results/"

models_embedding = {"embgemma": "EmbGemma", "qwen3emb": "Qwen3Emb"}   # zero-shot: nothing is trained, emissions = encoding
embedding_params = {"EmbGemma": 300e6, "Qwen3Emb": 600e6}              # fixed sizes of the pre-trained models
models_trained = {"BOW2ITM": our_model, "USEM2ITM": "USEM", "BERT2ITM": "BERT"}
efficiency_models = [our_model, "USEM", "BERT", "EmbGemma", "Qwen3Emb"]  # models shown in the efficiency figure
model_colors.update({"EmbGemma": "#9C27B0", "Qwen3Emb": "#795548", "kNN-BoW": "#607D8B"})


def read_rows(path):
    with open(path) as f:
        return list(csv.DictReader(f))


def resolve_md5(model, dataset, subset):
    """md5 of the checkpoint behind the paper tables for (model, dataset, subset)."""
    for r in read_rows(results_path + "tripadvisor_retrain_md5.csv"):
        if (r["model"], r["dataset"], r["subset"]) == (model, dataset, subset):
            return r["new_md5"]
    for r in read_rows(results_path + "user_agg_summary_known.csv"):
        if (r["model"], r["dataset"], r["subset"]) == (model, dataset, subset):
            return r["md5"]
    raise KeyError((model, dataset, subset))


def count_checkpoint_params(model_path):
    """Number of weights of a saved Keras checkpoint (same count as model.count_params(), without building the model)."""
    import tensorflow as tf
    ck = tf.train.load_checkpoint(model_path + "weights")
    return int(sum(int(np.prod(shape)) if shape else 1 for k, shape in ck.get_variable_to_shape_map().items()
                   if k.endswith("/.ATTRIBUTES/VARIABLE_VALUE") and "optimizer" not in k and "save_counter" not in k))


In [ ]:
def extract_baseline_data(path, dataset, subset):
    
    def process_section(lines):
        # Get the columns
        columns = lines[1].split("|")
        columns = [col.strip() for col in columns]
        
        # Get the data
        data = []
        for line in lines[3:]:
            if line.strip() == '':
                continue
            row = line.split("|")
            row = [item.strip() for item in row]
            data.append(row)
        
        # Create the DataFrame
        df = pd.DataFrame(data, columns=columns)
        return df

    # Get the file with the times
    paths = sorted(Path(path).iterdir(), key=os.path.getmtime)
    filepath = [str(f) for f in paths if "CornacExp" in f.name][-1]

    # Read the file
    with open(filepath, 'r') as file: lines = file.readlines()

    # Split the sections
    validation_start = lines.index('VALIDATION:\n')
    test_start = lines.index('TEST:\n')

    validation_lines = lines[validation_start + 1:test_start]
    test_lines = lines[test_start + 1:]

    # Process each section
    validation_df = process_section(validation_lines)
    test_df = process_section(test_lines)
    
    test_df.rename(columns={"":"Model", "Train (s)":"Train_time"}, inplace=True)
    
    model_paper_names = {"GridSearch_EASEᴿ":"EASEᴿ", "GridSearch_BPR":"BPR", "online_ibpr": "IBPR"}
    
    # Get every model's test results
    all_test_results = pd.read_csv(path+"results.csv").reset_index(drop=True)
    all_test_results = all_test_results.merge(test_df[["Model", "Train_time"]], on="Model", how="left")
    all_test_results["Model"] = all_test_results["Model"].apply(lambda x: model_paper_names[x] if x in model_paper_names.keys() else x)
    
    all_test_results.insert(0, "Subset", subset)
    all_test_results.insert(0, "Set", dataset)
    return all_test_results

def evaluate_cold_start(model_class):
    # Remove users that appear in TRAIN
    train_dev_users = model_class.DATASET.DATA["TRAIN_DEV"].userId.unique()
    # original_test_len = len(model_class.DATASET.DATA["TEST"])
    model_class.DATASET.DATA["TEST"] = model_class.DATASET.DATA["TEST"][~model_class.DATASET.DATA["TEST"]["userId"].isin(train_dev_users)]
    model_class.DATASET.DATA["TEST"] = model_class.DATASET.DATA["TEST"].drop_duplicates(subset=["userId", "id_item"], keep='last', inplace=False)
    # Evaluate on the test set
    test_ret = model_class.train(dev=False, save_model=True)
    # Evaluate on the test set
    test_ret = model_class.evaluate(test=True, verbose=False)
    return test_ret.to_dict(orient="records")[0]

### Obtain number of test users in all/known/cold scenarios

In [ ]:
 users_result = []
 for dataset, subsets in datasets.items():
        for subset in subsets:
            # Define the file name
            base_path = "/media/nas/pperez/code/2024/ETeR-X/"
            path = f"{base_path}{baselines_path}/{dataset}/{subset}/"
            # Load the first model (they all give the same result)
            model_class = load_best_model(model=list(models_cold.keys())[0], dataset=dataset, subset=subset, base_path=base_path)
            # Get the number of test users
            test_total_users = len(model_class.DATASET.DATA["TEST"]["userId"].unique())
            # Get the number of cold-start test users
            train_dev_users = model_class.DATASET.DATA["TRAIN_DEV"]["userId"].unique()
            test_cold_users = len(model_class.DATASET.DATA["TEST"][~model_class.DATASET.DATA["TEST"]["userId"].isin(train_dev_users)]["userId"].unique())
            # Get the number of test users already seen in train-dev
            test_known_users = len(model_class.DATASET.DATA["TEST"][model_class.DATASET.DATA["TEST"]["userId"].isin(train_dev_users)]["userId"].unique())
            # Print info
            users_result.append((set_names[dataset.title()], subset_names[subset.title()], test_total_users, test_cold_users, test_known_users))

In [ ]:
experiment_user_info = pd.DataFrame(users_result, columns=["Set", "Subset", "All", "Cold", "Known"])
experiment_user_info.pivot_table(index=["Set", "Subset"], values=["All", "Known", "Cold"]).transpose()

### Extract all experiment results

In [ ]:
def collect_model_data():
    """Efficiency data (time, parameters, epochs, emissions in g of CO2) of the paper checkpoints + zero-shot embeddings."""
    emb = {(r["model"], r["dataset"], r["subset"]): r for r in read_rows(results_path + "embedding_results.csv")}
    rows = []
    for dataset, subsets in datasets.items():
        for subset in subsets:
            base = dict(Set=set_names[dataset.title()], Subset=subset_names[subset.title()])
            for model_key, name in models_trained.items():
                md5 = resolve_md5(model_key, dataset, subset)
                path = f"models/{model_key}/{dataset}/{subset}/{md5}/"
                log = pd.read_csv(path + "log.csv")
                # emissions_train.csv stores the CUMULATIVE emissions (kg) of the final training run: the last row is the total
                co2_g = 1000 * pd.read_csv(path + "emissions_train.csv")["emissions"].iloc[-1]
                rows.append({**base, "Model": name, "Train_time": log["e_time"].sum(), "Params": count_checkpoint_params(path),
                             "Epochs": log["epoch"].max() + 1, "co2": co2_g, "md5": md5})
            for key, name in models_embedding.items():
                e = emb[(key, dataset, subset)]
                rows.append({**base, "Model": name, "Train_time": float(e["encode_time_s"]), "Params": embedding_params[name],
                             "Epochs": 0, "co2": float(e["emissions_g"]), "md5": ""})
    return pd.DataFrame(rows)


def collect_metrics(cold=False):
    """NDCG@10/Recall@10 per model/dataset from the result files of the paper.
    cold=False: users seen in training, aggregated per user (paper Table 5); cold=True: unseen users, per review (Section 4.3.3)."""
    rows = []
    def add(model, dataset, subset, ndcg, recall):
        rows.append({"Set": set_names[dataset.title()], "Subset": subset_names[subset.title()], "Model": model, "NDCG@10": float(ndcg), "Recall@10": float(recall)})
    if not cold:
        for r in read_rows(results_path + "user_agg_summary_known.csv"):
            if r["model"] == "BOW2ITM" or (r["dataset"] == "amazon" and r["model"] in models_trained):
                add(models_trained[r["model"]], r["dataset"], r["subset"], r["NDCG@10_user"], r["Recall@10_user"])
        for f, m in (("user_agg_summary_known_usem_resplit.csv", "USEM"), ("user_agg_summary_known_bert_resplit.csv", "BERT")):
            for r in read_rows(results_path + f):
                if r["dataset"] != "amazon":
                    add(m, r["dataset"], r["subset"], r["NDCG@10_user"], r["Recall@10_user"])
        for r in read_rows(results_path + "knn_bow_results_known.csv"):
            add("kNN-BoW", r["dataset"], r["subset"], r["ndcg10_user"], r["recall10_user"])
        for r in read_rows(results_path + "embedding_results_known.csv"):
            add(models_embedding[r["model"]], r["dataset"], r["subset"], r["ndcg10_user"], r["recall10_user"])
    else:
        for f, trained in (("review_eval_known_vs_cold.csv", ("BOW2ITM", "amazon")), ("review_eval_known_vs_cold_usem_resplit.csv", None), ("review_eval_known_vs_cold_bert_resplit.csv", None)):
            for r in read_rows(results_path + f):
                if r["partition"] != "cold":
                    continue
                if trained is not None and not (r["model"] == "BOW2ITM" or r["dataset"] == "amazon"):
                    continue
                add(models_trained[r["model"]], r["dataset"], r["subset"], r["ndcg10"], r["recall10"])
        for r in read_rows(results_path + "knn_bow_results_coldstart.csv"):
            add("kNN-BoW", r["dataset"], r["subset"], r["ndcg10"], r["recall10"])
        for r in read_rows(results_path + "embedding_results_coldstart.csv"):
            add(models_embedding[r["model"]], r["dataset"], r["subset"], r["ndcg10"], r["recall10"])
        for r in read_rows(results_path + "mostpop_coldstart_results.csv"):
            add("MostPop", r["dataset"], r["subset"], r["ndcg10_cold"], r["recall10_cold"])
    return pd.DataFrame(rows)


if USE_PAPER_CHECKPOINTS:
    efficiency = collect_model_data()
    # Cold-start table: efficiency + per-review cold metrics (users unseen in training)
    all_cold_start_data = efficiency.merge(collect_metrics(cold=True), on=["Set", "Subset", "Model"], how="outer")
    # Baseline table: collaborative-filtering baselines from the Baselines logs + the rest of the models (known users, per user)
    cf_rows = []
    for dataset, subsets in datasets.items():
        for subset in subsets:
            cf = extract_baseline_data(f"{baselines_path}/{dataset}/{subset}/", dataset=set_names[dataset.title()], subset=subset_names[subset.title()])
            cf["Model"] = cf["Model"].apply(lambda x: models_baseline[x] if x in models_baseline.keys() else x)
            cf_rows.append(cf[cf["Model"].isin(["MostPop", "EASEᴿ", "BiVAE", "BPR"])])
    all_baseline_data = pd.concat(cf_rows + [efficiency.merge(collect_metrics(cold=False), on=["Set", "Subset", "Model"], how="outer")], ignore_index=True)
    efficiency.to_excel("output/efficiency_data.xlsx", index=False)   # inspectable copy of the numbers behind the efficiency figure

else:
    # ----- Original behaviour (kept for reference): best_models.csv checkpoints, cached in models/all_*.xlsx -----
    if not os.path.exists("models/all_baseline_data.xlsx") and not os.path.exists("models/all_cold_start_data.xlsx"):

        all_baseline_data = []
        all_cold_start_data = []

        for dataset, subsets in datasets.items():
            for subset in subsets:
                # Define the file name
                base_path = "/media/nas/pperez/code/2024/ETeR-X/"
                path = f"{base_path}{baselines_path}/{dataset}/{subset}/"
                # Extract all the baseline data
                baseline_data = extract_baseline_data(path, dataset=set_names[dataset.title()], subset=subset_names[subset.title()])
                # Rename them
                baseline_data["Model"] = baseline_data["Model"].apply(lambda x: models_baseline[x] if x in models_baseline.keys() else x)
                # Remove irrelevant models
                baseline_data = baseline_data[baseline_data["Model"].isin(list(models_baseline.values()))]
                # Do the same for the non-baseline models
                cold_start_data = []
                for model in list(models_cold.keys()):
                    # Load the best model
                    model_class = load_best_model(model=model, dataset=dataset, subset=subset, base_path=base_path)
                    # Get the cold_start results
                    metrics = evaluate_cold_start(model_class)
                    # Get the number of parameters
                    params = model_class.MODEL.count_params()
                    # Get the training time
                    log_data = pd.read_csv(model_class.MODEL_PATH+"log.csv")
                    time = log_data["e_time"].sum()
                    # Get the epochs
                    epochs = log_data["epoch"].max()+1
                    # Get the emissions
                    emissions = pd.read_csv(model_class.MODEL_PATH+"emissions_train.csv").drop_duplicates("project_name")
                    emissions = emissions["emissions"].sum()
                    # Add to baselines
                    baseline_data.loc[baseline_data["Model"]==models_cold[model], "Epochs"] = epochs
                    baseline_data.loc[baseline_data["Model"]==models_cold[model], "Train_time"] = time
                    baseline_data.loc[baseline_data["Model"]==models_cold[model], "Params"] = params
                    baseline_data.loc[baseline_data["Model"]==models_cold[model], "co2"] = emissions
                    # Add to cold_start
                    cold_start_data.append({"Set":set_names[dataset.title()], "Subset":subset_names[subset.title()], "Model": models_cold[model], "Train_time": time, "Params":params, "Epochs":epochs, "co2": emissions, **metrics})
                    # Free RAM
                    del model_class
                
                # Add to the corresponding final datasets
                all_baseline_data.extend(baseline_data.to_dict(orient='records'))
                all_cold_start_data.extend(cold_start_data)

        all_baseline_data = pd.DataFrame(all_baseline_data)
        all_cold_start_data = pd.DataFrame(all_cold_start_data)

        # Save the results, since recomputing them is time-consuming
        all_baseline_data.to_excel("models/all_baseline_data.xlsx", index=False)
        all_cold_start_data.to_excel("models/all_cold_start_data.xlsx", index=False)

    else:
        # Save the results, since recomputing them is time-consuming
        all_baseline_data = pd.read_excel("models/all_baseline_data.xlsx")
        all_cold_start_data = pd.read_excel("models/all_cold_start_data.xlsx")


order_all = list(models_baseline.values()) + ["kNN-BoW", "EmbGemma", "Qwen3Emb"]
order_cold = list(models_cold.values()) + ["kNN-BoW", "EmbGemma", "Qwen3Emb"]
for df, order in ((all_baseline_data, order_all), (all_cold_start_data, order_cold)):
    for col in ("Train_time", "Params", "Epochs", "co2"):          # the CF logs store some of these as text
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Set"] = pd.Categorical(df["Set"], list(set_names.values()))
    df["Subset"] = pd.Categorical(df["Subset"], list(dict.fromkeys(subset_names.values())))
    df["Model"] = pd.Categorical(df["Model"], list(dict.fromkeys(order)))


### Export experiment results

In [ ]:
bl_results = all_baseline_data.melt(id_vars=['Set', 'Subset', 'Model'], value_vars=['NDCG@10', 'Recall@10'], var_name='Metric', value_name='Value')
bl_results = bl_results.pivot_table(index=['Set', 'Subset', 'Metric'], columns='Model', values='Value')
bl_results.to_excel("output/baselines_results.xlsx")

In [ ]:
cs_results = all_cold_start_data.melt(id_vars=['Set', 'Subset', 'Model'], value_vars=['NDCG@10', 'Recall@10'], var_name='Metric', value_name='Value')
cs_results = cs_results.pivot_table(index=['Set', 'Subset', 'Metric'], columns='Model', values='Value')
cs_results.to_excel("output/cold_start_results.xlsx")

### Create tables with unique parameter information and times


In [ ]:
time_results = all_baseline_data[all_baseline_data['Train_time'].notna()].pivot_table(index=["Set", "Subset"], columns=["Model"], values=["Train_time"])
param_results = all_cold_start_data.pivot_table(index=["Set", "Subset"], columns=["Model"], values=["Params"])

time_results.to_excel("output/time_results.xlsx")
param_results.to_excel("output/param_results.xlsx")

# 2. Generate graphics for the paper

In [ ]:
def bar_plot_comparison(data, column_name="co2", title="Total train emissions (gr)", share_y=False, ylim=None, legend=True, y_scale="linear"):
    # Sort by model
    data = data.sort_values(["Set", "Subset", "Model"])
    # Compute the number of elements on X for each dataset
    num_elements = data[["Set", "Subset"]].drop_duplicates().groupby("Set", sort=False)["Subset"].size().values.tolist()
    sets = data["Set"].unique()
    models = data["Model"].unique()

    # Normalize the subplot widths so they sum to 1
    total_elements = sum(num_elements)
    relative_widths = [n / total_elements for n in num_elements]

    # Create a figure with horizontal subplots and custom widths
    fig, axes = plt.subplots(1, len(sets), figsize=(sum(num_elements) * .65+2, 2.8), gridspec_kw={'width_ratios': relative_widths}, sharey=share_y)

    # Iterate over the datasets and their corresponding axes
    for plot_idx, (dataset, ax) in enumerate(zip(sets, axes)):
        subset_data = data[data["Set"]==dataset].sort_values("Subset")
        X = subset_data["Subset"].unique().tolist()
        model_values = subset_data.groupby("Model", sort=False).apply(lambda x: x[column_name].values.tolist()).to_frame().to_dict()[0]

        X_axis = np.arange(len(X))
        
        bar_width = 0.7 / len(model_values)  # Adjust the bar width
        offset = 0  # Initialize the offset
        margin = 0.02
        # model_names = {"BOW2ITM":"TeRCEx", "USEM2ITM":"USEM", "BERT2ITM":"BERT"}
        
        for i, (model, values) in enumerate(model_values.items()):
            color = model_colors[model] # model_colors[i % len(model_colors)]  # Pick a color from the list based on the model's position
            ax.bar(X_axis + offset - margin, values, bar_width-margin, label=model, color=color)
            offset += bar_width  # Increase the offset for the next model

        ax.set_xticks(X_axis + bar_width * (len(model_values) - 1) / 2)
        ax.set_xticklabels(X)  # Adjust the X axis labels

        # if plot_idx == 0: ax.set_ylabel(axis_label)
        if share_y and ylim is not None: ax.set_ylim(ylim)
        ax.set_title(dataset)
        ax.set_axisbelow(True)
        ax.grid(axis="y")
        ax.set_yscale(y_scale)
    
    # Show the legend only once
    if legend:
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.1), ncol=len(models))

    # Adjust the layout so the subplots don't overlap
    # plt.title(dataset)
    fig.suptitle(title, fontsize=15, fontweight="bold", y=0.95)
    plt.tight_layout()

    plt.savefig(f"output/bar_plot_{column_name}.pdf",bbox_inches='tight')
    plt.show()
    
    return fig

bar_plot_comparison(all_baseline_data, column_name="Train_time", y_scale="log", share_y=True, title="Training time (s)")

# Remove MostPop, for these plots it doesn't add much
cold_no_pop = all_cold_start_data.loc[all_cold_start_data["Model"]!="MostPop"].reset_index(drop=True)
cold_no_pop = cold_no_pop.loc[cold_no_pop["Model"].isin(efficiency_models)].reset_index(drop=True)
cold_no_pop["Model"] = pd.Categorical(cold_no_pop["Model"], efficiency_models)
cold_no_pop = cold_no_pop.sort_values(["Set", "Subset"])

fig1 = bar_plot_comparison(cold_no_pop, column_name="co2", y_scale="log", share_y=True, title="Train emissions (g of CO2)")
fig2 = bar_plot_comparison(cold_no_pop, column_name="Params", y_scale="log", share_y=True, title="Model parameters")
# Epochs only make sense for the models that are actually trained (not for the zero-shot embeddings)
trained_only = cold_no_pop.loc[cold_no_pop["Model"].isin([our_model, "USEM", "BERT"])].copy()
trained_only["Model"] = pd.Categorical(trained_only["Model"], [our_model, "USEM", "BERT"])
fig2 = bar_plot_comparison(trained_only, column_name="Epochs", y_scale="log", share_y=True, title="Model epochs")

### Version of the previous one that allows stacking of several figures


In [ ]:
def bar_plot_comparison(data, axes, column_name="co2", title="Total train emissions (gr)", share_y=False, ylim=None, legend=True, show_title=True, show_x_labels=True, y_scale="linear"):
    # Sort by model
    data = data.sort_values(["Set", "Subset", "Model"])
    # Compute the number of elements on X for each dataset
    num_elements = data[["Set", "Subset"]].drop_duplicates().groupby("Set", sort=False)["Subset"].size().values.tolist()
    sets = data["Set"].unique()
    models = data["Model"].unique()

    # Normalize the subplot widths so they sum to 1
    total_elements = sum(num_elements)
    relative_widths = [n / total_elements for n in num_elements]

    # Iterate over the datasets and their corresponding axes
    for plot_idx, (dataset, ax) in enumerate(zip(sets, axes.flatten())):
        X = data[data["Set"]==dataset]["Subset"].unique().tolist()
        model_values = data[data["Set"]==dataset].groupby("Model", sort=False).apply(lambda x: x[column_name].values.tolist()).to_frame().to_dict()[0]

        X_axis = np.arange(len(X))
        
        bar_width = 0.7 / len(model_values)  # Adjust the bar width
        offset = 0  # Initialize the offset
        margin = 0.02
        
        for i, (model, values) in enumerate(model_values.items()):
            color = model_colors[model]
            try:
                ax.bar(X_axis + offset - margin, values, bar_width-margin, label=model, color=color)
            except:
                print(model, values, color, dataset)
            offset += bar_width  # Increase the offset for the next model

        if show_x_labels:
            ax.set_xticks(X_axis + bar_width * (len(model_values) - 1) / 2)
            ax.set_xticklabels(X)  # Adjust the X axis labels
        else:
            ax.set_xticks([])
        
        if plot_idx==0: ax.set_ylabel(title)

        if share_y and ylim is not None: ax.set_ylim(ylim)
        if show_title: ax.set_title(dataset)
        ax.set_axisbelow(True)
        ax.grid(axis="y")
        ax.set_yscale(y_scale)
    
    return ax

data = cold_no_pop
num_elements = data[["Set", "Subset"]].drop_duplicates().groupby("Set", sort=False)["Subset"].size().values.tolist()
sets = data["Set"].unique()
models = data["Model"].unique()

# Normalize the subplot widths so they sum to 1
total_elements = sum(num_elements)
relative_widths = [n / total_elements for n in num_elements]

fig, ax = plt.subplots(2, 3, figsize=(sum(num_elements) * .6+1, 3.5), gridspec_kw={'width_ratios': relative_widths}, sharey='row')

bar_plot_comparison(cold_no_pop, ax[0,:], column_name="co2", y_scale="log", share_y=True, show_x_labels=False, title="Emissions (g CO$_2$)")
bar_plot_comparison(cold_no_pop, ax[1,:], column_name="Params", y_scale="log", share_y=True, show_title=False, title="Parameters")

handles, labels = ax[0,0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.025), ncol=len(labels))

plt.tight_layout()
fig.subplots_adjust(wspace=0.1, hspace=0.1, top=0.85, bottom=0.15)

plt.savefig(f"output/bar_plot_co2_params.pdf",bbox_inches='tight')

plt.show()

In [ ]:
def polar_plot_comparison(data, metrics=["NDCG@10", "Recall@10"], output_file="polar_plot.pdf"):
    
    def cerrar_ciclo(data):
        return np.append(data, data[0])
    
    def plot_radial_data(axis, models, datasets, plot_data, column, title="Title", max_y=None, fmt='{:.3f}'):
        axis.set_theta_zero_location("N")
        axis.set_theta_direction(-1)

        for idx, modelo in enumerate(models):
            values = plot_data.loc[plot_data["Model"] == modelo][column].values
            values = np.append(values, values[0])
            theta = np.linspace(0, 2 * np.pi, len(values), endpoint=True)
            color = model_colors[modelo]  # Pick a color from the list based on the model's position
            axis.fill(theta, values, alpha=0.1, color=color)
            axis.plot(theta, values, label=modelo, marker=".",  color=color)
        axis.set_title(title, pad=30, fontweight="bold")
        
        if max_y is not None: axis.set_ylim(0, max_y)
        axis.yaxis.set_major_locator(FixedLocator(axis.get_yticks()))
        axis.set_yticklabels([fmt.format(tick) for tick in axis.get_yticks()], ha='center', va='bottom', color='lightgrey', fontweight='bold')
        axis.set_axisbelow(True)
        # Get radial labels away from plotted line
        axis.set_rlabel_position(180)  
        axis.set_xticks(np.linspace(0, 2*np.pi, len(datasets), endpoint=False))
        axis.set_xticklabels(datasets, fontsize=10)
        # Adjust grid colors and style
        axis.grid(linewidth=1, linestyle = '-', color="lightgrey")
        axis.yaxis.grid(linewidth=0.5, color="lightgrey")
        # Adjust datasets labels, rotation, color, ...
        angles = np.linspace(0, -2*np.pi, len(axis.get_xticklabels()) + 1)
        angles = np.rad2deg(angles[:-1])
        labels = []
        for label, angle in zip(axis.get_xticklabels(), angles):
            x, y = label.get_position()
            dataset_text = label.get_text()
            dataset_color = "darkred" if "POI" in dataset_text else "navy" if "RST" in dataset_text else "darkslategray"
            # Adjust the labels' position and rotate them so they form a 90-degree angle with the radius
            if angle <= -90 and angle >= -270: rotation_angle = angle  - 180
            else: rotation_angle = angle
            lab = axis.text(x, y + 0.035, dataset_text, transform=label.get_transform(), ha='center', va='center', rotation=rotation_angle, color=dataset_color)
            labels.append(lab)
        axis.set_xticklabels([])

    # List of models
    modelos = list(data["Model"].sort_values().unique())
    # Build a list of set+subset combos as titles
    datasets = data[["Set", "Subset"]].drop_duplicates().apply(lambda x: x["Set"]+"-"+x["Subset"], axis=1).values.tolist()
    # Count sets and subsets
    n_datasets = len(datasets)
    # Create the two radial charts (polar plots)
    fig, axs = plt.subplots(1, 2, subplot_kw=dict(polar=True), figsize=(8,5))
    axs = axs.flatten()
    
    for id_plot, metric in enumerate(metrics):
        plot_radial_data(axs[id_plot], modelos, datasets, data, fmt='{:.1f}',max_y=1, column=metric, title=metric)
        
    # Add a single legend at the bottom center
    lines, labels = axs[0].get_legend_handles_labels()
    fig.legend(lines, labels, loc='lower center', bbox_to_anchor=(0.5, 0.15), ncol=1)
    # Adjust the horizontal spacing between the plots
    plt.tight_layout()
    fig.subplots_adjust(wspace=0.6, hspace=0, top=0.85, bottom=0.15)
    # Save the figure
    plt.savefig("output/"+output_file, bbox_inches='tight')
    plt.show()

polar_plot_comparison(all_baseline_data, metrics=["NDCG@10", "Recall@10"], output_file="baseline_polar_plot.pdf")
polar_plot_comparison(all_cold_start_data, metrics=["NDCG@10", "Recall@10"], output_file="cold_start_polar_plot.pdf")